In [ ]:
## Customer Dimension Table (Olist Dataset)
## This notebook prepares the customer dataset for use as a dimension table in an analytical data model.

# The goal is to:
# - Clean and standardize customer identifiers
# - Remove duplicates and inconsistencies
# - Create a reliable customer dimension table ('dim_customers')

# This table will later be joined with order data to enrich transactional analysis.

In [ ]:
## Setup for file load

In [ ]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

In [ ]:
## Initial Data Exploration

In [ ]:
customers.head()

In [ ]:
customers.shape

In [ ]:
customers.isna().sum()

In [ ]:
## Data Quality Checks

In [ ]:
customers.duplicated().sum()

In [ ]:
customers['customer_id'].is_unique

In [ ]:
customers['customer_id'].nunique()

In [ ]:
## Handling Customer Identity

# Customers may appear multiple times due to multiple orders.

# To create a clean dimension table:
# - We group by 'customer_unique_id'
# - Ensure one row per unique customer
# - Retain stable attributes like location

In [ ]:
duplicate_customers = customers[customers.duplicated(['customer_unique_id'], keep=False)]

In [ ]:
duplicate_customers.shape

In [ ]:
duplicate_customers.head(20)

In [ ]:
customers['customer_unique_id'].nunique()

In [ ]:
## Inspected duplicate customer_unique_id records
customers[customers.duplicated(['customer_unique_id'], keep=False)]

In [ ]:
## Check if any customers appear in multiple states
state_counts = customers.groupby('customer_unique_id')['customer_state'].nunique()
state_counts[state_counts > 1].sum()

In [ ]:
## Create Customer Dimension Table

In [ ]:
dim_customers = customers.drop_duplicates(subset=['customer_unique_id'])

In [ ]:
dim_customers = dim_customers.drop(columns=['customer_id'])


In [ ]:
## State Standardization
# Customer state abbreviations are mapped to full names for improved readability in analysis.

In [ ]:
state_map = {
'AC': 'Acre',
'AL': 'Alagoas',
'AP': 'Amapá',
'AM': 'Amazonas',
'BA': 'Bahia',
'CE': 'Ceará',
'DF': 'Distrito Federal',
'ES': 'Espírito Santo',
'GO': 'Goiás',
'MA': 'Maranhão',
'MT': 'Mato Grosso',
'MS': 'Mato Grosso do Sul',
'MG': 'Minas Gerais',
'PA': 'Pará',
'PB': 'Paraíba',
'PR': 'Paraná',
'PE': 'Pernambuco',
'PI': 'Piauí',
'RJ': 'Rio de Janeiro',
'RN': 'Rio Grande do Norte',
'RS': 'Rio Grande do Sul',
'RO': 'Rondônia',
'RR': 'Roraima',
'SC': 'Santa Catarina',
'SP': 'São Paulo',
'SE': 'Sergipe',
'TO': 'Tocantins'
}

In [ ]:
dim_customers['customer_state_name'] = dim_customers['customer_state'].map(state_map)

In [ ]:
## State Mapping Validation

In [ ]:
dim_customers['customer_state_name'].isna().sum()

In [ ]:
## Customer Dimension Validation

In [ ]:
dim_customers['customer_unique_id'].is_unique

In [ ]:
dim_customers.shape

In [ ]:
## Final Customer Dimension

# The resulting dataset ('dim_customers') contains:
# - One row per unique customer
# - Cleaned and standardized attributes
# - A stable key ('customer_unique_id') for joining with fact tables

# This table serves as the customer dimension in the star schema.

In [ ]:
## Save cleaned customer dimension table
dim_customers.to_csv("../data/clean/dim_customers.csv", index=False)